Se importan las librerías necesarias para el análisis de datos, incluyendo herramientas para manipulación de datos y visualización.

In [2]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sn

Cargamos el archivo original en excel

In [4]:
df = pd.read_excel(r"AlmacenesRFM.xlsx")

Exploración inicial

Se analiza la estructura del dataset para identificar el tipo de datos, la cantidad de registros y posibles valores nulos.

Observaciones El dataset contiene múltiples columnas con diferentes niveles de completitud. Se detecta que no todas las filas corresponden a órdenes completas, lo que sugiere la presencia de datos adicionales como detalles de productos

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16009 entries, 0 to 16008
Data columns (total 30 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   OrderId                                  3675 non-null   float64
 1   StoreId                                  3675 non-null   float64
 2   OrderGuid                                16009 non-null  object 
 3   CustomerId                               16009 non-null  object 
 4   OrderStatusId                            16009 non-null  object 
 5   PaymentStatusId                          16009 non-null  object 
 6   ShippingStatusId                         16009 non-null  object 
 7   OrderSubtotalInclTax                     16009 non-null  object 
 8   OrderSubtotalExclTax                     16009 non-null  object 
 9   OrderSubTotalDiscountInclTax             16009 non-null  object 
 10  OrderSubTotalDiscountExclTax             16009

In [7]:
df.head(10)

,OrderId,StoreId,OrderGuid,CustomerId,OrderStatusId,PaymentStatusId,ShippingStatusId,OrderSubtotalInclTax,OrderSubtotalExclTax,OrderSubTotalDiscountInclTax,...,CurrencyRate,CustomerCurrencyCode,AffiliateId,PaymentMethodSystemName,ShippingPickupInStore,ShippingMethod,ShippingRateComputationMethodSystemName,CustomValuesXml,VatNumber,CreatedOnUtc
0,3955.0,1.0,c962edb6-4278-4d60-a79c-7ab6c9c48327,2413329,30,10,20,77825.8,68872.39,0,...,1.0,CRC,0.0,Payments.PayInStore,0.0,RETIRAR EN SUCURSAL HATILLO,Shipping.FixedByWeightByTotal,NaN,NaN,46143.114836
1,NaN,NaN,Name,Sku,PriceExclTax,PriceInclTax,Quantity,DiscountExclTax,DiscountInclTax,TotalExclTax,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,COCINA PARA GAS PORTATIL 1 QUEMADOR TRUPER,39833,13495.5,15249.915,1,1499.5,1694.435,13495.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,LAMPARA LED FLEXIBLE 1W CON USB TRUPER,24908,968.4,1094.292,2,215.2,243.176,1936.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,"PULIDORA ELECTRICA 6 "" 900W TRUPER",45189,53440.09,60387.3017,1,5937.788,6709.7004,53440.09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,3954.0,1.0,f2bcb1e4-b54b-4b8e-8f2a-ce263cc57396,3017399,30,10,20,27746.08,24554.05,0,...,1.0,CRC,0.0,Payments.P2P,0.0,ENVÍO A DOMICILIO,Shipping.FixedByWeightByTotal,NaN,NaN,46142.888532
6,NaN,NaN,Name,Sku,PriceExclTax,PriceInclTax,Quantity,DiscountExclTax,DiscountInclTax,TotalExclTax,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,ESMERIL RECTO 1/4 INDUST 600W ESRE-1/4N TR...,24024,24554.05,27746.0765,1,2728.228,3082.8976,24554.05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,3953.0,1.0,11150913-2707-4794-9cf1-1c41b830ad5b,3017397,40,10,20,241994.04,214154.02,0,...,1.0,CRC,0.0,Payments.P2P,0.0,RETIRAR EN SUCURSAL ESQUINA,Shipping.FixedByWeightByTotal,NaN,NaN,46142.831666
9,NaN,NaN,Name,Sku,PriceExclTax,PriceInclTax,Quantity,DiscountExclTax,DiscountInclTax,TotalExclTax,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Problema identificado

Se observa que el dataset contiene filas que no representan órdenes completas, sino detalles de productos asociados a cada orden.

Estas filas incluyen valores como "Name", "Sku" y precios unitarios, lo que indica que el dataset mezcla información a nivel de orden y a nivel de producto.

Para el análisis RFM, solo se requieren datos a nivel de orden, por lo que es necesario filtrar estas filas.

Limpieza de datos

Se filtran las filas que corresponden únicamente a órdenes válidas, eliminando aquellas que contienen información de productos.

Esto se logra seleccionando únicamente las filas donde OrderId no es nulo.

Validación

Después del filtrado, se verifica que el dataset contenga únicamente órdenes válidas, eliminando registros inconsistentes.

In [9]:
df = df[df["OrderId"].notna()]

In [10]:
df.head(10)

,OrderId,StoreId,OrderGuid,CustomerId,OrderStatusId,PaymentStatusId,ShippingStatusId,OrderSubtotalInclTax,OrderSubtotalExclTax,OrderSubTotalDiscountInclTax,...,CurrencyRate,CustomerCurrencyCode,AffiliateId,PaymentMethodSystemName,ShippingPickupInStore,ShippingMethod,ShippingRateComputationMethodSystemName,CustomValuesXml,VatNumber,CreatedOnUtc
0,3955.0,1.0,c962edb6-4278-4d60-a79c-7ab6c9c48327,2413329,30,10,20,77825.8,68872.39,0,...,1.0,CRC,0.0,Payments.PayInStore,0.0,RETIRAR EN SUCURSAL HATILLO,Shipping.FixedByWeightByTotal,NaN,NaN,46143.114836
5,3954.0,1.0,f2bcb1e4-b54b-4b8e-8f2a-ce263cc57396,3017399,30,10,20,27746.08,24554.05,0,...,1.0,CRC,0.0,Payments.P2P,0.0,ENVÍO A DOMICILIO,Shipping.FixedByWeightByTotal,NaN,NaN,46142.888532
8,3953.0,1.0,11150913-2707-4794-9cf1-1c41b830ad5b,3017397,40,10,20,241994.04,214154.02,0,...,1.0,CRC,0.0,Payments.P2P,0.0,RETIRAR EN SUCURSAL ESQUINA,Shipping.FixedByWeightByTotal,NaN,NaN,46142.831666
11,3952.0,1.0,5ecbeb85-f689-4fa1-8c26-9630f18d9f58,3017397,10,10,20,435589.28,385477.24,0,...,1.0,CRC,0.0,Payments.P2P,0.0,RETIRAR EN SUCURSAL ESQUINA,Shipping.FixedByWeightByTotal,NaN,NaN,46142.829075
14,3951.0,1.0,71f62770-447a-4f8c-b43b-d6dea4ed07e4,28796,20,10,20,82129.21,72680.72,0,...,1.0,CRC,0.0,Payments.PayInStore,0.0,RETIRAR EN SUCURSAL HATILLO,Shipping.FixedByWeightByTotal,NaN,NaN,46142.798291
18,3950.0,1.0,61b13b9c-e073-4f60-98a3-9e6448b05998,28796,40,10,20,82129.21,72680.72,0,...,1.0,CRC,0.0,Payments.P2P,0.0,RETIRAR EN SUCURSAL HATILLO,Shipping.FixedByWeightByTotal,NaN,NaN,46142.796269
22,3949.0,1.0,84ca6711-4255-4186-aab4-776e5861292f,3017200,30,30,20,228657.27,202351.57,0,...,1.0,CRC,0.0,Payments.P2P,0.0,RETIRAR EN SUCURSAL ESQUINA,Shipping.FixedByWeightByTotal,NaN,NaN,46142.742326
26,3948.0,1.0,bfbe4b8e-9b24-4510-99b2-ed9703afc188,3017095,30,10,20,21706.94,19209.68,0,...,1.0,CRC,0.0,Payments.PayInStore,0.0,RETIRAR EN SUCURSAL ESQUINA,Shipping.FixedByWeightByTotal,NaN,NaN,46142.738588
30,3947.0,1.0,810661b3-5dbf-4ec6-b3b9-b7c6b19031f3,1849422,20,10,20,243083.93,215118.52,0,...,1.0,CRC,0.0,Payments.CheckMoneyOrder,0.0,RETIRAR EN SUCURSAL HATILLO,Shipping.FixedByWeightByTotal,NaN,NaN,46142.657066
33,3946.0,1.0,1a3e616f-0a22-4bf5-8adb-75c43fc2df5b,1329066,30,10,20,59740.24,52867.47,0,...,1.0,CRC,0.0,Payments.PayInStore,0.0,RETIRAR EN SUCURSAL CENTRAL,Shipping.FixedByWeightByTotal,NaN,NaN,46142.551092


In [11]:
df["OrderTotal"].head()

0      77825.80
5      27746.08
8     241994.04
11    435589.28
14     82129.21
Name: OrderTotal, dtype: float64

In [12]:
df["OrderTotal"].isnull().sum()

0

In [13]:
df["OrderTotal"].describe()

count    3.675000e+03
mean     6.459117e+04
std      1.590534e+05
min      0.000000e+00
25%      1.259156e+04
50%      2.957547e+04
75%      6.389870e+04
max      3.869266e+06
Name: OrderTotal, dtype: float64

In [14]:
df[df["OrderTotal"] == 0]

,OrderId,StoreId,OrderGuid,CustomerId,OrderStatusId,PaymentStatusId,ShippingStatusId,OrderSubtotalInclTax,OrderSubtotalExclTax,OrderSubTotalDiscountInclTax,...,CurrencyRate,CustomerCurrencyCode,AffiliateId,PaymentMethodSystemName,ShippingPickupInStore,ShippingMethod,ShippingRateComputationMethodSystemName,CustomValuesXml,VatNumber,CreatedOnUtc
3356,3148.0,1.0,617ed979-4fc6-44e6-9469-30d33529da3a,2243954,40,30,20,0,0,0,...,1.0,CRC,0.0,NaN,0.0,ENVÍO A DOMICILIO,Shipping.FixedByWeightByTotal,NaN,NaN,45959.747481
3359,3147.0,1.0,c22ebf7b-d0c9-4134-9d2e-850386b7cf1b,2647438,40,30,20,0,0,0,...,1.0,CRC,0.0,NaN,0.0,RETIRAR EN SUCURSAL ESQUINA,Shipping.FixedByWeightByTotal,NaN,NaN,45959.741282


In [15]:
df=df[df["OrderTotal"] > 0]

In [16]:
df["OrderTotal"].describe()

count    3.673000e+03
mean     6.462634e+04
std      1.590896e+05
min      3.550000e+00
25%      1.259204e+04
50%      2.957547e+04
75%      6.395727e+04
max      3.869266e+06
Name: OrderTotal, dtype: float64

Validación de órdenes

Se identificaron órdenes con valor total igual a cero, asociadas probablemente a compras canceladas o no completadas.

Dado que el análisis RFM busca estudiar comportamiento real de compra, estas órdenes fueron excluidas para evitar distorsiones en las métricas de frecuencia y valor monetario.

Limpieza y transformación de fechas

Se identificó que la columna CreatedOnUtc se encontraba en formato numérico, lo que impedía su uso directo como fecha.

Durante la exploración, se detectó que estos valores correspondían a un formato no estándar derivado de Excel, por lo que fue necesario realizar una conversión manual.

Proceso aplicado

Se ajustaron los valores restando 25569 días para alinear el origen de fechas de Excel con el formato Unix.
Se convirtió la columna a tipo datetime.
Finalmente, se simplificó la fecha eliminando la parte de tiempo, ya que para el análisis RFM solo es relevante el día.
Esto permite utilizar correctamente la información temporal para calcular la métrica de Recency.

In [19]:
df["CreatedOnUtc"].head(10)

0     46143.114836
5     46142.888532
8     46142.831666
11    46142.829075
14    46142.798291
18    46142.796269
22    46142.742326
26    46142.738588
30    46142.657066
33    46142.551092
Name: CreatedOnUtc, dtype: float64

In [20]:
df["CreatedOnUtc"] = pd.to_datetime(df["CreatedOnUtc"]- 25569,
                                 unit="D")

In [21]:
df["CreatedOnUtc"].head(10)

0    2026-05-01 02:45:21.825000017
5    2026-04-30 21:19:29.161000240
8    2026-04-30 19:57:35.968000060
11   2026-04-30 19:53:52.044999817
14   2026-04-30 19:09:32.347999877
18   2026-04-30 19:06:37.608000180
22   2026-04-30 17:48:56.971999912
26   2026-04-30 17:43:34.036000075
30   2026-04-30 15:46:10.521000261
33   2026-04-30 13:13:34.327000061
Name: CreatedOnUtc, dtype: datetime64[ns]

In [22]:
# Formatear para quitar los decimales
df["CreatedOnUtc"] = df["CreatedOnUtc"].dt.strftime("%Y-%m-%d %H:%M:%S")

In [23]:
df["CreatedOnUtc"].head(10)

0     2026-05-01 02:45:21
5     2026-04-30 21:19:29
8     2026-04-30 19:57:35
11    2026-04-30 19:53:52
14    2026-04-30 19:09:32
18    2026-04-30 19:06:37
22    2026-04-30 17:48:56
26    2026-04-30 17:43:34
30    2026-04-30 15:46:10
33    2026-04-30 13:13:34
Name: CreatedOnUtc, dtype: object

Selección de variables relevantes

Para el análisis RFM, no es necesario utilizar todas las columnas del dataset.

Se seleccionan únicamente las variables clave:

CustomerId: identificación del cliente
OrderId: identificación de la orden (para medir frecuencia)
OrderTotal: monto total de la compra (para medir valor monetario)
CreatedOnUtc: fecha de la orden (para medir recencia)
Esto permite simplificar el análisis y enfocarse en la información relevante.

In [25]:
df = df[[
    "CustomerId",
    "OrderId",
    "OrderTotal",
    "CreatedOnUtc"
]]

Renombramos las columnas

In [27]:
df = df.rename(columns={"CustomerId": "ClienteID",
                       "OrderId": "PedidoID",
                        "OrderTotal": "TotalOrden",
                        "CreatedOnUtc": "Fecha"})

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3673 entries, 0 to 16006
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ClienteID   3673 non-null   object 
 1   PedidoID    3673 non-null   float64
 2   TotalOrden  3673 non-null   float64
 3   Fecha       3673 non-null   object 
dtypes: float64(2), object(2)
memory usage: 143.5+ KB


In [29]:
df["Fecha"].head()

0     2026-05-01 02:45:21
5     2026-04-30 21:19:29
8     2026-04-30 19:57:35
11    2026-04-30 19:53:52
14    2026-04-30 19:09:32
Name: Fecha, dtype: object

In [30]:
df["Fecha"] = pd.to_datetime(df["Fecha"],errors="coerce")

UltimaFecha = df["Fecha"].max()

rfm = df.groupby("ClienteID").agg({
    "Fecha": "max",
    "PedidoID": "count",
    "TotalOrden": "sum"
})

rfm.columns= ["UltimaCompra","Frecuencia","Monto"]

rfm["Recencia"]= (UltimaFecha - rfm["UltimaCompra"]).dt.days

Construcción del modelo RFM

Para analizar el comportamiento de los clientes, se construyó el modelo RFM (Recency, Frequency, Monetary), una técnica ampliamente utilizada en análisis de negocio.

Métricas utilizadas

Recencia: número de días desde la última compra del cliente.
Frecuencia: cantidad total de compras realizadas por el cliente.
Monto: valor total gastado por el cliente.
Metodología

Se agruparon los datos por cliente (ClienteID) para calcular:

La fecha de la última compra (max de Fecha)
El número de órdenes (count de PedidoID)
El total gastado (sum de TotalOrden)
Posteriormente, se calculó la métrica Recencia utilizando la fecha más reciente del dataset como referencia.

Este enfoque permite resumir el comportamiento de cada cliente en tres dimensiones clave para su análisis.

In [32]:
rfm.head()

,UltimaCompra,Frecuencia,Monto,Recencia
ClienteID,,,,
1,2024-10-05 03:09:51,3,752558.48,572
31,2024-06-07 17:11:23,1,1436.61,692
104,2026-04-29 21:46:17,324,55721975.03,1
107,2026-04-09 14:46:50,28,1875466.70,21
28614,2024-12-28 02:52:50,17,53498.35,488


Resultados iniciales del análisis RFM

Tras calcular las métricas de recencia, frecuencia y valor monetario, se identificaron diferencias significativas en el comportamiento de compra entre clientes.

Algunos clientes presentan altos niveles de frecuencia y gasto acumulado, lo que indica una fuerte relación comercial con la empresa. Por otro lado, también se identificaron clientes con largos períodos sin realizar compras, reflejados en valores altos de recencia.

El análisis permitió detectar clientes de alto valor, clientes recurrentes y clientes potencialmente inactivos, proporcionando una base sólida para futuras estrategias de segmentación, retención y fidelización.

In [34]:
rfm["Puntos_Recencia"] = pd.qcut(rfm["Recencia"], 5, labels=[5,4,3,2,1])
rfm["Puntos_Frecuencia"] = pd.qcut(rfm["Frecuencia"].rank(method = "first"), 5, labels=[1,2,3,4,5])
rfm["Puntos_Monto"] = pd.qcut(rfm["Monto"],5, labels=[1,2,3,4,5])

In [35]:
rfm["Puntaje_RFM"]= (
    rfm["Puntos_Recencia"].astype(str) +
    rfm["Puntos_Frecuencia"].astype(str) +
    rfm["Puntos_Monto"].astype(str)
    
)

In [36]:
def segment(row):
    if row["Puntos_Recencia"] == 5 and row["Puntos_Frecuencia"] >= 4 :
        return "VIP"
    elif row["Puntos_Recencia"]  >=4 :
        return "Leal"
    elif row ["Puntos_Recencia"] < 2 :
        return "En Riesgo"
    else :
        return "Regular"

In [37]:
rfm["Segmentación"] = rfm.apply(segment, axis=1)

In [38]:
rfm["Segmentación"].value_counts()

Segmentación
Regular      824
Leal         445
En Riesgo    414
VIP          384
Name: count, dtype: int64

In [39]:
rfm["Monto"] = (
    rfm["Monto"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

rfm["Monto"] = pd.to_numeric(rfm["Monto"], errors="coerce")

rfm["Monto"] = rfm["Monto"].round(2)

In [40]:
rfm.to_excel("rfm_customers_clean.xlsx")

In [41]:
tendencia_ventas = df[["Fecha", "TotalOrden"]]

In [42]:
tendencia_ventas.to_excel("tendencia_ventas.xlsx", index=False)

Resultados de la segmentación de clientes

A partir del modelo RFM se clasificaron los clientes en distintos segmentos según su comportamiento de compra.

La mayor parte de los clientes pertenece al segmento "Regular", lo que indica una base amplia de clientes con actividad promedio dentro del negocio.

También se identificó un grupo importante de clientes "Leales", caracterizados por mantener compras frecuentes y recientes, representando una base sólida para estrategias de retención.

El segmento "VIP" agrupa clientes de alto valor, con altos niveles de frecuencia y gasto acumulado, siendo los clientes más importantes para el negocio.

Por otro lado, se detectó un número considerable de clientes "En Riesgo", los cuales presentan largos períodos sin comprar, lo que representa una oportunidad para desarrollar campañas de reactivación y recuperación.

Insight clave

Aunque los clientes VIP representan un grupo más pequeño en cantidad, su impacto económico y nivel de actividad los convierte en un segmento estratégico para la empresa.

Asimismo, la presencia significativa de clientes en riesgo sugiere la necesidad de implementar estrategias orientadas a mejorar la retención y disminuir la pérdida de clientes activos.

 Conclusiones

A partir del análisis RFM, se logró segmentar a los clientes según su comportamiento de compra, permitiendo identificar patrones importantes dentro del negocio y comprender mejor el nivel de actividad y valor de cada grupo de clientes.

Hallazgos principales

- El segmento más grande corresponde a clientes **"Regular"** con 824 clientes, lo que representa una amplia base de consumidores con comportamiento promedio y potencial de crecimiento mediante estrategias de fidelización.

- Se identificaron 445 clientes dentro del segmento **"Leal"**, caracterizados por mantener compras frecuentes y relativamente recientes, representando una base sólida de clientes recurrentes.

- El segmento **"VIP"** está compuesto por 384 clientes de alto valor, con altos niveles de frecuencia y gasto acumulado, convirtiéndose en el grupo más importante para la rentabilidad del negocio.

- También se detectaron 414 clientes **"En Riesgo"**, lo que indica la existencia de clientes que anteriormente mostraban actividad, pero que actualmente presentan largos períodos sin comprar.

Recomendaciones

- Implementar campañas de reactivación dirigidas a clientes **"En Riesgo"** para incentivar nuevas compras y reducir la pérdida de clientes.

- Diseñar estrategias de fidelización para clientes **"Regular"** con el objetivo de aumentar su frecuencia de compra y convertirlos en clientes recurrentes.

- Ofrecer beneficios exclusivos y programas de recompensa para clientes **"VIP"**, buscando mantener su lealtad y maximizar su valor a largo plazo.

- Analizar el comportamiento de los clientes **"Leal"** para fortalecer su relación con la empresa y mantener su nivel de actividad.

Valor del análisis

El modelo RFM permitió transformar datos transaccionales en información estratégica y accionable, facilitando la identificación de clientes valiosos, clientes recurrentes y clientes con riesgo de abandono.

Este tipo de análisis puede servir como base para futuras estrategias comerciales orientadas a mejorar la retención de clientes, optimizar campañas de marketing y maximizar los ingresos del negocio.